# LPI Phase 3A — iLINCS API v2
Consulta reproducible con selección explícita de bibliotecas: **LIB_6 = genetic/CGS knockdown** y **LIB_5 = chemical perturbagens**.

La firma LPI está congelada: 6 genes UP y 14 genes DOWN. No se modifica según los resultados.


In [ ]:
import requests, pandas as pd, io, json, zipfile, os, time

UP = ['BCL2L11', 'CDKN1B', 'GADD45A', 'SESN3', 'SOD2', 'CAT']
DOWN = ['GHR', 'IGF1', 'IGF1R', 'IRS1', 'IRS2', 'PIK3CA', 'PIK3CB', 'PDPK1', 'AKT1', 'AKT2', 'MTOR', 'RPTOR', 'RPS6KB1', 'EIF4EBP1']

lines = ['Name_GeneSymbol\tValue_LogDiffExp']
lines += [f'{g}\t1' for g in UP]
lines += [f'{g}\t-1' for g in DOWN]
sig_text = '\n'.join(lines) + '\n'
open('LPI_signature.tsv','w',encoding='utf-8').write(sig_text)
print(sig_text)


In [ ]:
upload_url = 'https://www.ilincs.org/api/SignatureMeta/upload'
with open('LPI_signature.tsv','rb') as fh:
    resp = requests.post(upload_url, files={'file': ('LPI_signature.tsv', fh, 'text/tab-separated-values')}, timeout=180)
resp.raise_for_status()
uj = resp.json()
print(json.dumps(uj, indent=2)[:3000])
sf = uj.get('status',{}).get('fileName')
if isinstance(sf, list): sf = sf[0]
if not sf:
    raise RuntimeError('No se pudo recuperar status.fileName de la respuesta de upload.')
print('Archivo temporal iLINCS:', sf)


In [ ]:
def concordances(lib):
    url = 'https://www.ilincs.org/api/ilincsR/findConcordances'
    r = requests.post(url, data={'file': sf, 'lib': lib}, timeout=240)
    r.raise_for_status()
    j = r.json()
    tab = j.get('concordanceTable', [])
    return pd.json_normalize(tab), j

cgs, cgs_raw = concordances('LIB_6')
chem, chem_raw = concordances('LIB_5')
print('CGS/LIB_6:', len(cgs), 'filas')
print('Chemical/LIB_5:', len(chem), 'filas')
display(cgs.head())
display(chem.head())
cgs.to_csv('LPI_iLINCS_CGS_LIB6.csv', index=False)
chem.to_csv('LPI_iLINCS_chemical_LIB5.csv', index=False)


In [ ]:
def enrichment(lib, metadata=True):
    url = 'https://www.ilincs.org/api/ilincsR/signatureEnrichment'
    params = {'sigFile': sf, 'library': lib}
    if metadata:
        params['metadata'] = 'TRUE'
    r = requests.get(url, params=params, timeout=240)
    r.raise_for_status()
    j = r.json()
    tab = j.get('enrichment', [])
    return pd.json_normalize(tab), j

gen_enr, gen_raw = enrichment('LIB_6', metadata=True)
chem_enr, chem_enr_raw = enrichment('LIB_5', metadata=True)
print('Genetic perturbation enrichment:', len(gen_enr))
print('Chemical perturbation enrichment:', len(chem_enr))
display(gen_enr.head())
display(chem_enr.head())
gen_enr.to_csv('LPI_iLINCS_genetic_perturbation_enrichment.csv', index=False)
chem_enr.to_csv('LPI_iLINCS_chemical_perturbation_enrichment.csv', index=False)


In [ ]:
for name,obj in [('CGS_LIB6_raw.json', cgs_raw),('chemical_LIB5_raw.json', chem_raw),('genetic_enrichment_raw.json', gen_raw),('chemical_enrichment_raw.json', chem_enr_raw)]:
    with open(name,'w',encoding='utf-8') as f:
        json.dump(obj,f,ensure_ascii=False,indent=2)
files_out = ['LPI_signature.tsv','LPI_iLINCS_CGS_LIB6.csv','LPI_iLINCS_chemical_LIB5.csv','LPI_iLINCS_genetic_perturbation_enrichment.csv','LPI_iLINCS_chemical_perturbation_enrichment.csv','CGS_LIB6_raw.json','chemical_LIB5_raw.json','genetic_enrichment_raw.json','chemical_enrichment_raw.json']
with zipfile.ZipFile('LPI_iLINCS_results_v2.zip','w',zipfile.ZIP_DEFLATED) as z:
    for fn in files_out:
        if os.path.exists(fn): z.write(fn)
print('ZIP creado:', os.path.getsize('LPI_iLINCS_results_v2.zip'), 'bytes')


In [ ]:
try:
    from google.colab import files
    files.download('LPI_iLINCS_results_v2.zip')
except Exception as e:
    print('Fuera de Colab: toma LPI_iLINCS_results_v2.zip de la carpeta de trabajo.')
    print(e)
